# PC to z to CAD

In this notebook I want to develop the pipeline from point cloud to CAD-model. To do that I will use my trained PointNet++ to encode a point cloud into the latent represenation z and then the DeepCAD decoder will reconstruct the CAD model from this.

In [88]:
import sys
import importlib
import os
import torch
import open3d as o3d

sys.path.append(os.path.abspath("../code"))
import dataset
from dataset import PointCloudEmbeddingDataset
importlib.reload(dataset)

<module 'dataset' from '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/code/dataset.py'>

In [89]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

In [90]:
train_dataset = PointCloudEmbeddingDataset("../data", 'train')

### Loading train dataset ###

Number of samples that should be in the train set: 161240
Files on disk: 160982 --> There are 258 missing point cloud files in the train set.

Checking latent representation:
All latent represenations are valid.

--- DONE ---



In [91]:
index = 123

point_cloud = train_dataset[index][0]
latent_rep = train_dataset[index][1]
train_dataset.get_path(index)

'../data/pc_cad/0056/00566921.ply'

In [92]:
def visualize_pc(pc_path):
    point_cloud = o3d.io.read_point_cloud(pc_path)
    print(point_cloud)
    o3d.visualization.draw_geometries([point_cloud])
#visualize_pc(train_dataset.get_path(index))

In [93]:
ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
LINE_IDX = ALL_COMMANDS.index('Line')
ARC_IDX = ALL_COMMANDS.index('Arc')
CIRCLE_IDX = ALL_COMMANDS.index('Circle')
EOS_IDX = ALL_COMMANDS.index('EOS')
SOL_IDX = ALL_COMMANDS.index('SOL')
EXT_IDX = ALL_COMMANDS.index('Ext')

## Plan
- import trained PN++ DONE
- infer PC with PN++ DONE
- compare predicted z with target z DONE (using MSE)
- import pretrained decoder
- infer z with decoder
- compare predicted CAD-sequence with target CAD-sequence
- visualize CAD model

## Load trained PointNet++

In [94]:
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

classifier = model.get_model(256, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

model_path = 'best.pth'
model_dict = torch.load(model_path, map_location=torch.device(device), weights_only=True)
state_dict = model_dict['model_state_dict']
classifier.load_state_dict(state_dict)
classifier = classifier.to(device)
classifier.eval()

print("Loaded model")

Loaded model


## Convert Point Cloud to Latent Representation

In [95]:
def pc_to_z(model, pc, target):
    with torch.no_grad():
        pc = pc.unsqueeze(0)
        target = target.unsqueeze(0)
        pc = pc.transpose(2,1)
        pred, _ = classifier(pc)
        mse = criterion(pred,target)
        print(f"MSE: {mse:.6f}")
        return pred
        

In [96]:
z = pc_to_z(classifier, point_cloud, latent_rep)
print(z.shape)
z = z.unsqueeze(0)
print(z.shape)

MSE: 0.108668
torch.Size([1, 256])
torch.Size([1, 1, 256])


## Load DeepCAD model

In [97]:
sys.path.append(os.path.abspath(".."))
from models.DeepCAD.trainer.trainerAE import TrainerAE
from models.DeepCAD.config.configAE_jupyter import ConfigAE

In [98]:
cfg = ConfigAE('test')

----Experiment Configuration-----
proj_dir             ../data/latent
data_root            data
exp_name             pretrained
gpu_ids              0
batch_size           512
num_workers          8
nr_epochs            1000
lr                   0.001
grad_clip            1.0
warmup_step          2000
cont                 False
ckpt                 1000
vis                  False
save_frequency       500
val_frequency        10
vis_frequency        2000
augment              False
mode                 None
outputs              None
z_path               None


## Convert z into CAD-Sequence

In [99]:
tr_agent = TrainerAE(cfg)
tr_agent.load_ckpt(cfg.ckpt)
tr_agent.net.eval()
with torch.no_grad():
    output = tr_agent.decode(z)
    batch_out_vec = tr_agent.logits2vec(output)

Loading checkpoint from ../data/latent/pretrained/model/ckpt_epoch1000.pth ...


In [100]:
for k, v in output.items():
    print(f"{k}: {v.shape}")
print(f"batch_out_vec: {out_vec.shape}")

command_logits: torch.Size([1, 60, 6])
args_logits: torch.Size([1, 60, 16, 257])
batch_out_vec: (60, 17)


In [101]:
out_vec = batch_out_vec[0]
print(out_vec.shape)
out_command = out_vec[:, 0]
print(out_command.shape)
seq_len = out_command.tolist().index(EOS_IDX)
data = out_vec[:seq_len]
print(data.shape)

(60, 17)
(60,)
(6, 17)


## Convert CAD-Sequence to CAD-Model

In [102]:
import h5py
import numpy as np
save_path = os.path.join(os.getcwd(), '{}.h5'.format("example"))
with h5py.File(save_path, 'w') as fp:
    fp.create_dataset('out_vec', data=data, dtype=np.int64)